# Lesson 02 - Exploring Microsoft Agent Framework

Di **Microsoft Agent Framework (MAF)** na one unified framework for building AI agents. E get correct, composable architecture wey get four core building blocks:

- **Client** – e connect to AI model endpoint and e dey handle communication
- **Agent** – e wrap client wit instructions and tool definitions
- **Tools** – dem dey extend agent capabilities wit custom functions wea di model fit call
- **Session** – e dey maintain conversation history for multi-turn interactions

For dis lesson, we go build **travel booking agent** wey go check destination availability using these concepts.


## Setup


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Understanding the Agent Framework Architecture

Di Microsoft Agent Framework follow one kain layered architecture:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – An `AzureAIProjectAgentProvider` dey connect to Azure OpenAI deployment. E dey handle authentication, request formatting, and response parsing.
2. **Agent** – E dey create from di client through `provider.create_agent()`, di agent dey combine model access with instructions (system prompt) and tools.
3. **Tools** – Na Python functions wey dem put `@tool` for, wey di agent fit invoke to perform actions or collect data.
4. **Session** – An `AgentSession` object (wey dem create via `agent.create_session()`) wey dey store conversation history, so that multi-turn dialogue fit happen where di agent sabi wetin dem talk before.

Make we build each layer step by step.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Adding Tools with the @tool Decorator

Tools dey allow agents to do tin dem pass just to generate text. The `@tool` decorator dey turn regular Python function to somtin wey the agent fit call.

Key points:
- Use `Annotated[type, "description"]` so the model fit understand each parameter.
- The docstring go become the tool description wey the model go see.
- `approval_mode="never_require"` mean say the tool go run automatically without user confirmation.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Creating an Agent wit Tools

Now we combine di client, instructions, and tools into one agent. Di `instructions` dey act as di system prompt — dem dey define di agent pesona and how e go dey behave.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Multi-Turn Conversations wit Sessions

An `AgentSession` (wey dem create via `agent.create_session()`) dey keep track of all messages wey dey conversation. By passing the same session to each `agent.run()` call, the agent fit access the full conversation history and fit refer back to earlier messages.

We dey pass `tools=[check_destination_availability]` so the agent fit call our availability checker during each turn.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Summary

For dis lesson you explore di four main tins wey Microsoft Agent Framework get:

| Concept | Wetin You Learn |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` dey connect to Azure OpenAI wit credential-based auth |
| **Agent** | `provider.create_agent()` dey join model connection wit instructions and name |
| **Tools** | Di `@tool` decorator dey show Python functions for di agent to call |
| **Session** | `agent.create_session()` dey keep conversation history as e dey happen |

Dem build tins like dis join body to create agents wey fit hold natural talks, call external functions, and remember context — dis na the base for more advanced agentic patterns wey dem go talk for later lessons.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:  
Dis document na AI translation service wey dem call [Co-op Translator](https://github.com/Azure/co-op-translator) translate am. Even though we try make am correct, abeg sabi say machine translation fit get some mistake or no too correct. The original document wey dem write for im own language na the correct one wey you suppose trust. If na serious matter, e better make person wey sabi language do the translation. We no go responsible if anybody misunderstand or use dis translation the wrong way.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
